# HyperSense v1.2 — Regional Adaptation

## Stage 2: Ghana + Nigeria Model Development

### Research Question

Does incorporating Nigerian data into training improve a parsimonious six-feature hypertension model, compared with the Ghana-only HyperSense v1.1 model tested as-is in Stage 1?This stage isolates the effect of adding more local data while keeping the feature set fixed. Stage 3 will evaluate whether adding further features provides additional benefit.

### Background

In Stage 1, the Ghana-derived HyperSense v1.1 model was externally evaluated in the Nigerian REMAH cohort without retraining or recalibration. Discrimination decreased from an AUC of 0.792 in Ghana to 0.747 in the full REMAH cohort, while specificity fell from 0.576 to 0.407 and calibration indicated systematic overprediction. One likely contributor was the `scale_pos_weight` strategy tuned for Ghana's relatively low hypertension prevalence, which did not transfer well to the REMAH population.

Stage 2 therefore incorporates Nigerian data directly into model development. The class-imbalance strategy and decision threshold will be re-derived using the pooled Ghana + REMAH training data rather than inherited from v1.1.

### Outcome

The outcome remains the same measurement-only hypertension definition used in v1.1 and Stage 1. Hypertension is defined as:

- mean SBP ≥140 mmHg, **and/or**
- mean DBP ≥90 mmHg.

All five available blood-pressure readings are required for the outcome calculation (`skipna=False`). Antihypertensive treatment (`HTN_DR`) is not included in the outcome definition.

This preserves the same target construct across Ghana and REMAH and allows the datasets to be pooled validly.

### Predictor Features

The six predictors remain locked to the HyperSense v1.1 feature set:

1. `age`
2. `gender`
3. `residence`
4. `educational_level`
5. `tobacco_use`
6. `bmi`

The harmonization mappings established in Stage 1 will be reused unchanged for REMAH. No additional predictors are introduced at this stage.

### Evaluation Strategy

The pooled Ghana + REMAH dataset will be split into training and test sets using the same stratified 80/20 approach used in v1.1. The resulting test set will be evaluated overall and separately by data source:

- **Pooled test set:** overall performance after regional adaptation.
- **REMAH-origin test subset:** the primary comparison with Stage 1 transportability performance.
- **Ghana-origin test subset:** a sanity check that adaptation does not substantially degrade performance in the original population.

The final model will use the same XGBoost model family and a comparable `RandomizedSearchCV` tuning strategy with internal 10-fold cross-validation. `scale_pos_weight` will be excluded from the hyperparameter search space entirely. Instead, the decision threshold will be re-derived from the pooled training data.

The primary question is whether incorporating Nigerian data improves performance, particularly whether the specificity collapse observed during Stage 1 transportability is reduced in the REMAH-origin test subset.

### 1. Package Versions

In [1]:
# Record package versions for reproducibility

import pandas as pd
import numpy as np
import sklearn
import xgboost

print(f"pandas:   {pd.__version__}")
print(f"numpy:    {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"xgboost:  {xgboost.__version__}")

pandas:   2.3.3
numpy:    2.4.0
scikit-learn: 1.8.0
xgboost:  3.3.0


### 2. Imports

In [2]:
# Core libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    brier_score_loss,
    precision_score,
    recall_score,
)
from sklearn.calibration import calibration_curve
from sklearn.base import clone

# XGBoost
from xgboost import XGBClassifier

# Reproducibility
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

### Load the datasets

In [3]:
# Load the Ghana and Nigerian datasets

ghana = pd.read_csv("../outputs/hypersense_v2_data.csv")
nigeria = pd.read_csv("../outputs/transportability/remah_harmonized.csv")

print(f"Ghana dataset:   {len(ghana):,} rows")
print(f"Nigeria dataset: {len(nigeria):,} rows")

Ghana dataset:   7,844 rows
Nigeria dataset: 4,089 rows


In [4]:
# Confirm the required features and outcome are available in both datasets

model_features = [
    "age",
    "gender",
    "residence",
    "educational_level",
    "tobacco_use",
    "bmi",
]

outcome = "htn_status"

print("Ghana columns:")
print(ghana[model_features + [outcome]].columns.tolist())

print("\nNigeria columns:")
print(nigeria[model_features + [outcome]].columns.tolist())

Ghana columns:
['age', 'gender', 'residence', 'educational_level', 'tobacco_use', 'bmi', 'htn_status']

Nigeria columns:
['age', 'gender', 'residence', 'educational_level', 'tobacco_use', 'bmi', 'htn_status']


In [5]:
# Check completeness of the locked feature set

print("Ghana missing values:")
print(ghana[model_features + [outcome]].isna().sum())

print("\nNigeria missing values:")
print(nigeria[model_features + [outcome]].isna().sum())

Ghana missing values:
age                  0
gender               0
residence            0
educational_level    0
tobacco_use          0
bmi                  0
htn_status           0
dtype: int64

Nigeria missing values:
age                  0
gender               0
residence            0
educational_level    0
tobacco_use          0
bmi                  0
htn_status           0
dtype: int64


In [6]:
ghana.head()

,caseid,country,sample_weight,age,gender,residence,educational_level,tobacco_use,htn_status,bmi
0,1 1 1,Ghana,856663.0,50.0,1,0,2,0,1.0,1703.0
1,1 3 1,Ghana,856663.0,27.0,1,0,2,0,0.0,2048.0
2,1 6 1,Ghana,856663.0,24.0,1,0,2,0,0.0,2108.0
3,111 1,Ghana,856663.0,40.0,1,0,1,1,0.0,1972.0
4,119 1,Ghana,856663.0,43.0,1,0,2,0,0.0,2300.0


In [7]:
nigeria.head()

,age,gender,residence,educational_level,tobacco_use,bmi,htn_status
0,77.876712,1,0,1.0,1.0,18.832392,0.0
1,53.197260,1,1,1.0,0.0,30.323343,0.0
2,62.287671,1,0,2.0,0.0,19.486961,0.0
3,67.287671,1,0,1.0,0.0,20.202020,0.0
4,34.734247,1,0,3.0,0.0,29.239563,0.0


In [8]:
# Normalize Ghana BMI to match the Nigerian BMI scale

ghana["bmi"] = ghana["bmi"] / 100

print("Ghana BMI range after normalization:")
print(ghana["bmi"].agg(["min", "max"]).round(2))

Ghana BMI range after normalization:
min    13.34
max    54.35
Name: bmi, dtype: float64


In [9]:
# Confirm BMI is on the same scale in both datasets

print("BMI ranges:")
print(
    pd.DataFrame({
        "Ghana": ghana["bmi"].agg(["min", "max"]),
        "Nigeria": nigeria["bmi"].agg(["min", "max"])
    }).round(2)
)

BMI ranges:
     Ghana  Nigeria
min  13.34    11.67
max  54.35    56.30


In [10]:
# Add source labels before pooling

ghana["source"] = "Ghana"
nigeria["source"] = "Nigeria"

# Pool the two datasets while preserving source
pooled = pd.concat(
    [ghana, nigeria],
    axis=0,
    ignore_index=True
)

print(f"Combined dataset: {len(pooled):,} rows")

print("\nRows by source:")
print(pooled["source"].value_counts())

print("\nHypertension prevalence by source:")
print(
    pooled.groupby("source")["htn_status"]
    .mean()
    .mul(100)
    .round(1)
)

Combined dataset: 11,933 rows

Rows by source:
source
Ghana      7844
Nigeria    4089
Name: count, dtype: int64

Hypertension prevalence by source:
source
Ghana      11.5
Nigeria    28.2
Name: htn_status, dtype: float64


### 3. Pooled Train/Test Split

The pooled dataset is split into training and test sets using an 80/20 stratified split. Stratification is performed jointly on data source and hypertension status to preserve the Ghana/Nigeria composition and outcome distribution across both partitions.

In [11]:
# Create a combined stratification variable
pooled["stratify_group"] = (
    pooled["source"].astype(str)
    + "_"
    + pooled["htn_status"].astype(str)
)

# Stratified 80/20 train/test split
train_data, test_data = train_test_split(
    pooled,
    test_size=0.20,
    stratify=pooled["stratify_group"],
    random_state=RANDOM_STATE
)

# Remove the temporary stratification variable
train_data = train_data.drop(columns="stratify_group")
test_data = test_data.drop(columns="stratify_group")

print(f"Training set: {len(train_data):,} rows")
print(f"Test set: {len(test_data):,} rows")

Training set: 9,546 rows
Test set: 2,387 rows


### 4. Source and Outcome Composition After Splitting

The source and hypertension-status distributions are checked separately in the training and test sets to confirm that both partitions retain comparable Ghana/Nigeria composition and outcome prevalence.

In [12]:
# Report source composition within each split

for split_name, split_data in [
    ("Training", train_data),
    ("Test", test_data)
]:
    print(f"{split_name} set — source composition:")
    print(
        split_data["source"]
        .value_counts()
        .rename_axis("source")
        .to_frame("n")
    )

    print("\nSource proportion:")
    print(
        split_data["source"]
        .value_counts(normalize=True)
        .mul(100)
        .round(1)
        .rename_axis("source")
        .to_frame("percent")
    )

    print()

Training set — source composition:
            n
source       
Ghana    6275
Nigeria  3271

Source proportion:
         percent
source          
Ghana       65.7
Nigeria     34.3

Test set — source composition:
            n
source       
Ghana    1569
Nigeria   818

Source proportion:
         percent
source          
Ghana       65.7
Nigeria     34.3



In [13]:
# Check hypertension prevalence within each source and split

for split_name, split_data in [
    ("Training", train_data),
    ("Test", test_data)
]:
    print(f"{split_name} set — hypertension prevalence by source:")
    print(
        split_data.groupby("source")["htn_status"]
        .mean()
        .mul(100)
        .round(1)
        .rename("prevalence_percent")
    )

    print()

Training set — hypertension prevalence by source:
source
Ghana      11.5
Nigeria    28.2
Name: prevalence_percent, dtype: float64

Test set — hypertension prevalence by source:
source
Ghana      11.5
Nigeria    28.1
Name: prevalence_percent, dtype: float64

